# 02. Select four-month windows

Compare May–August and the existing green-up-aligned recipe without changing thresholds, labels or partitions. Uses the successful handoff selected by notebook 01, never the newest folder by modification time. If collection changed, start a new audit after it finishes.


**Input:** Verified quality.json from Preprocessing 01. **Output:** `windows.json`. **Next:** Preprocessing 03.

[Run order and reruns](../../docs/workflow.md).


In [5]:
from pathlib import Path
import sys
import json

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import preprocessing_workflow as workflow
from crop_pipeline.notebook_display import show_table
from crop_pipeline.windows import seasons

quality_input = ""  # Optional exact quality.json path from a new notebook 01 run; blank uses its verified pointer.


## Select and count qualifying windows

Existing successful outputs are verified and reused. Partial or changed outputs are preserved and require a new audit run. Counts here precede the per-parcel pixel cap. Satellite-estimated timing remains distinct from verified crop occupancy.


In [6]:
window_handoff = workflow.select(root, quality_input or None)
show_table(seasons.read_table(window_handoff.parent / "window_counts.csv"), title="Complete windows before pixel capping")
print("Window handoff:", window_handoff)
print("Organized outputs:", root / "pipeline_outputs/preprocessing" / window_handoff.parent.name / "windows")
print("Next: 03_feature_engineering.ipynb uses this verified handoff automatically.")


Detector outcomes: {'candidate': 2744, 'rejected': 3610, 'ambiguous': 1809}
Results: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/greenup
Reference-supported field-seasons: 0
Initial estimated proposals: 2276
Automatically qualified estimates: 1439
Total qualified estimated windows: 1439
Unresolved outcomes: 6724
Automatic timing failures: 837
Review templates: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/seasons/review
Reference-supported windows saved: 0
Estimated observation windows saved: 1439
Audit: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/seasons/season_review_audit.csv
Cycle selection: {'selected': 470, 'excluded_incomplete': 918, 'excluded_later': 51}
Selected estimated windows: 470


Window,Partition,Crop,Complete pixel windows before cap,Parcels
fixed,Training,Carrot,68463,273
fixed,Training,Cabbage,65582,422
fixed,Training,Potato,46190,379
fixed,Validation,Carrot,27183,96
fixed,Validation,Cabbage,16611,124
fixed,Validation,Potato,22970,156
fixed,Reserved test,Carrot,65,21
fixed,Reserved test,Cabbage,440,38
fixed,Reserved test,Potato,134,32
aligned,Training,Carrot,21837,109


Window handoff: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/windows.json
Organized outputs: /Users/josedavid/Documents/programming/external-crop-data-classification/pipeline_outputs/preprocessing/external_jqin0imq/windows
Next: 03_feature_engineering.ipynb uses this verified handoff automatically.


## Evidence and interpretation

Coverage describes this saved snapshot. Green-up is estimated timing; sample targets do not establish accuracy. See [research basis and limits](../../docs/research_basis.md) for sources and project-specific choices.


## Figures

Parcel groups with a complete fixed or aligned window, followed by up to three saved green-up detection examples. A green-up is an algorithmic estimate, not a verified planting date.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from crop_pipeline.visualization import workflow_figures as figures


def show_and_save(figure, folder, name):
    """Display inline and keep a PNG beside the run outputs for reports and slides."""
    print("Saved figure:", figures.save_figure(figure, folder, name))
    display(figure)
    plt.close(figure)

figure_folder = window_handoff.parent / "figures"
show_and_save(figures.draw_window_counts(figures.read_table(window_handoff.parent / "window_counts.csv")),
              figure_folder, "window_counts")
greenup_examples = sorted((window_handoff.parent / "greenup" / "plots").glob("*.png"))[:3]
for example in greenup_examples:
    print("Green-up example:", example)
    display(Image(filename=str(example)))
if not greenup_examples:
    print("No saved green-up plots for this run.")